# Does the linear fit improve with more PCs?

In [1]:
from astropy.io import fits

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from scipy import stats as ss
from scipy.io import readsav
from sklearn.decomposition import PCA

from tqdm import tqdm

import tools as t

## Load data

In [2]:
# load files
path_data = '../data/sim_series/'  # directory where the files are saved
N_files = 20  # 20 temporal series (from 0 to 19)
data_spectra = []
data_rv = []
for i in range(N_files):  # files have the same name, only number changes
    data_spectra.append(fits.getdata(path_data + f'flux_norm_6173_lionel_tau1_spatialscaling1_sansrotation_serie{i}.fits'))
    data_rv.append(readsav(path_data + f'res_rv_bis_6173_lionel_tau1_scaling1_sansrot_serie{i}.sav'))
data_spectra = np.array(data_spectra)
data_rv = np.array(data_rv)
data_rv[0].keys()

dict_keys(['rv_gauss', 'rv_correl', 'rv_deriv', 'xbiss', 'ybiss', 'l_true', 'lambd', 'flux_moy'])

In [6]:
c = 3e8  # speed of light [m/s]
# extract data
wl = np.array(data_rv[0]['lambd'])  # same wavelength grid for all time series (verified)
times = np.arange(0, len(data_spectra[0]) * 144, 144) / 60 / 24  # time in days, same for all time series
times_sec = times * 86400  # time in sec
wl = np.array(wl)

## Preprocessing

In [4]:
# calculate the mean spectra
mean_spec = np.array([np.mean(i, axis=0) for i in data_spectra])  # list of mean spectrum for each TS
d_S0 = np.array([np.gradient(i, wl) for i in mean_spec])  # list of derivative of the mean spectra for each TS

In [7]:
S_f = []  # array of matrices
D_vectors = []  # array of Doppler vectors

for i in range(N_files):  # for each TS
    s, v = t.subtract_D(wl, data_spectra[i], c=c)  # take out projection over the Doppler vector
    S_f.append(s)
    D_vectors.append(v)

S_f = np.array(S_f)
D_vectors = np.array(D_vectors)

## Calculate RV by template matching

In [5]:
# calculate radial velocity by template matching
C_0 = 1000  # continuum constant
rv_tm = []  # list of RV vector for each TS
for j in range(N_files):  # for each TS
    rv_j = []
    for k in range(len(times)):  # for each spectrum
        rv_j.append(t.calculate_rv(wl, mean_spec[j], data_spectra[j][k], C_0))  # calculate RV by TM
    rv_tm.append(np.array(rv_j))
rv_tm = np.array(rv_tm)

## PCA

In [8]:
# apply double centering to each S_f matrix
spectra_centered = np.array([t.double_centering(i) for i in S_f])

In [12]:
np.arange(10, 60, 5)

array([10, 15, 20, 25, 30, 35, 40, 45, 50, 55])

In [ ]:
# apply PCA to the centered matrix (use only TS1 to train the model)
n_components = np.arange(5, 65, 5)  # variable number of PCs

spec_0 = spectra_centered[0]  # only TS1

loadings = {}  # number of PCs: loadings
scores = {}  # number of PCs: scores
var = {}  # number of PCs: variances

for i in n_components:

    pca_flux_i = PCA(n_components=i)  # create PCA object
    pcs_i = pca_flux_i.fit_transform(spec_0)  # fit the model with spectra_centered and apply the dimensionality reduction
    
    loadings[i] = pca_flux_i.components_  # save loadings
    scores[i] = pcs_i.T  # save scores
    var[i] = pca_flux_i.explained_variance_ratio_  # save PC variation


In [ ]:
# use the scores to create a linear model
